In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

In [ ]:
DATASET_DIR = "../dataset_stage1"
REAL_LIFE_DIR = "../dataset_real_life_stage1"   # <-- put real camera captures here

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# Training schedule
EPOCHS_PHASE1 = 5
EPOCHS_PHASE2 = 15

LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-5

# Fine-tuning control
N_UNFROZEN = 40  # unfreeze last N layers of base_model (except BN)

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
def _random_jpeg_quality(x, min_q=40, max_q=95):
    x_uint8 = tf.cast(tf.clip_by_value(x, 0.0, 255.0), tf.uint8)
    q = tf.random.uniform([], min_q, max_q + 1, dtype=tf.int32)
    x_jpeg = tf.image.decode_jpeg(tf.image.encode_jpeg(x_uint8, quality=q), channels=3)
    return tf.cast(x_jpeg, tf.float32)

def _gaussian_kernel(ksize: int, sigma: float):
    ax = tf.range(-ksize // 2 + 1, ksize // 2 + 1, dtype=tf.float32)
    xx, yy = tf.meshgrid(ax, ax)
    kernel = tf.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel = kernel / tf.reduce_sum(kernel)
    kernel = kernel[:, :, tf.newaxis, tf.newaxis]  # (k, k, 1, 1)
    return kernel

def _gaussian_blur(x, ksize=5, sigma_min=0.3, sigma_max=1.5):
    sigma = tf.random.uniform([], sigma_min, sigma_max)
    kernel = _gaussian_kernel(ksize, sigma)
    x4 = x[tf.newaxis, ...]  # (1,H,W,C)
    kernel = tf.tile(kernel, [1, 1, tf.shape(x)[-1], 1])  # (k,k,C,1)
    x_blur = tf.nn.depthwise_conv2d(x4, kernel, strides=[1, 1, 1, 1], padding="SAME")
    return x_blur[0]

def _add_sensor_noise(x, std_min=0.0, std_max=12.0):
    std = tf.random.uniform([], std_min, std_max)
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=std, dtype=tf.float32)
    return tf.clip_by_value(x + noise, 0.0, 255.0)

def _random_crop_resize(x, y, min_scale=0.6, max_scale=1.0):
    h = tf.shape(x)[0]
    w = tf.shape(x)[1]
    scale = tf.random.uniform([], min_scale, max_scale)
    new_h = tf.cast(tf.cast(h, tf.float32) * scale, tf.int32)
    new_w = tf.cast(tf.cast(w, tf.float32) * scale, tf.int32)
    x = tf.image.random_crop(x, size=[new_h, new_w, 3], seed=SEED)
    x = tf.image.resize(x, IMG_SIZE, method="bilinear")
    return x, y

In [ ]:
def camera_like_augment(x, y):
    # x is float32 in [0,255]
    x, y = _random_crop_resize(x, y)

    # Horizontal flip is realistic; vertical flip usually isn't for phone photos
    x = tf.image.random_flip_left_right(x, seed=SEED)

    # --- 90° rotations sometimes (portrait/landscape/orientation issues)
    do_rot90 = tf.random.uniform([]) < 0.20
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x = tf.cond(do_rot90, lambda: tf.image.rot90(x, k=k), lambda: x)

    # --- tilt: small-angle rotation (5–20 degrees)
    # tf.image.rotate expects radians
    do_tilt = tf.random.uniform([]) < 0.60
    angle_deg = tf.random.uniform([], -20.0, 20.0)
    angle_rad = angle_deg * (np.pi / 180.0)
    x = tf.cond(do_tilt, lambda: tf.image.rotate(x, angles=angle_rad, fill_mode="reflect"), lambda: x)

    # Lighting & color shifts
    x = tf.image.random_brightness(x, max_delta=0.25)
    x = tf.image.random_contrast(x, lower=0.6, upper=1.4)
    x = tf.image.random_saturation(x, lower=0.7, upper=1.3)
    x = tf.image.random_hue(x, max_delta=0.05)

    # Blur sometimes (motion/defocus)
    do_blur = tf.random.uniform([]) < 0.35
    x = tf.cond(do_blur, lambda: _gaussian_blur(x, ksize=5), lambda: x)

    # Noise + JPEG artifacts sometimes
    do_noise = tf.random.uniform([]) < 0.50
    x = tf.cond(do_noise, lambda: _add_sensor_noise(x), lambda: x)

    do_jpeg = tf.random.uniform([]) < 0.35
    x = tf.cond(do_jpeg, lambda: _random_jpeg_quality(x), lambda: x)

    # Match backend preprocessing exactly (resize already done by dataset loader)
    x = preprocess_input(x)
    return x, y

def basic_preprocess(x, y):
    return preprocess_input(x), y

In [ ]:
train_ds = image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    os.path.join(DATASET_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_ds = image_dataset_from_directory(
    os.path.join(DATASET_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

real_life_ds = image_dataset_from_directory(
    REAL_LIFE_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

print("Train class_names:", train_ds.class_names)
print("Real-life class_names:", real_life_ds.class_names)

train_ds = train_ds.map(camera_like_augment, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(basic_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds.map(basic_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
real_life_ds = real_life_ds.map(basic_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
base_model = MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)
model = Model(inputs=base_model.input, outputs=output)

model.summary()

In [ ]:
def compile_model(lr: float):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
            tf.keras.metrics.Precision(name="precision", thresholds=0.5),
            tf.keras.metrics.Recall(name="recall", thresholds=0.5),

            # keep these extra recalls if you want to monitor them too
            tf.keras.metrics.Recall(name="recall_0.3", thresholds=0.3),
            tf.keras.metrics.Recall(name="recall_0.2", thresholds=0.2),
        ],
    )

In [ ]:
base_model.trainable = False
compile_model(LR_PHASE1)

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=3, restore_best_weights=True),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1
)

In [ ]:
base_model.trainable = True

# Freeze BatchNorm layers for stability
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Unfreeze only last N_UNFROZEN layers (excluding BN which we already froze)
for layer in base_model.layers[:-N_UNFROZEN]:
    layer.trainable = False

compile_model(LR_PHASE2)

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=2, min_lr=1e-7),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2
)

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    keys = set(h1.history.keys()) | set(h2.history.keys())
    for k in keys:
        merged[k] = list(h1.history.get(k, [])) + list(h2.history.get(k, []))
    return merged

merged = merge_histories(history1, history2)  # if you only have one phase, just use history2.history

plt.figure(figsize=(12, 7))

plt.subplot(2, 2, 1)
plt.plot(merged.get("accuracy", []), label="Train Accuracy")
plt.plot(merged.get("val_accuracy", []), label="Val Accuracy")
plt.legend(); plt.title("Accuracy")

plt.subplot(2, 2, 2)
plt.plot(merged.get("loss", []), label="Train Loss")
plt.plot(merged.get("val_loss", []), label="Val Loss")
plt.legend(); plt.title("Loss")

plt.subplot(2, 2, 3)
plt.plot(merged.get("precision", []), label="Train Precision")
plt.plot(merged.get("val_precision", []), label="Val Precision")
plt.legend(); plt.title("Precision")

plt.subplot(2, 2, 4)
plt.plot(merged.get("recall", []), label="Train Recall")
plt.plot(merged.get("val_recall", []), label="Val Recall")
plt.legend(); plt.title("Recall")

plt.tight_layout()
plt.show()

In [ ]:
def eval_stage1_three_way(ds, t_pos, t_neg, title="REAL-LIFE"):
    # Policy:
    # p >= t_pos => recyclable (1)
    # p <= t_neg => non-recyclable (0)
    # else => uncertain (-1)
    y_true = []
    y_prob = []
    for xb, yb in ds:
        p = model.predict(xb, verbose=0).reshape(-1)
        y_prob.append(p)
        y_true.append(tf.reshape(yb, [-1]).numpy())

    y_true = np.concatenate(y_true).astype(int)
    y_prob = np.concatenate(y_prob)

    pred3 = np.full_like(y_true, fill_value=-1, dtype=int)
    pred3[y_prob >= t_pos] = 1
    pred3[y_prob <= t_neg] = 0

    hard_fn = int(np.sum((pred3 == 0) & (y_true == 1)))  # recyclable -> non (worst)
    hard_fp = int(np.sum((pred3 == 1) & (y_true == 0)))
    uncertain_rec = int(np.sum((pred3 == -1) & (y_true == 1)))
    uncertain_non = int(np.sum((pred3 == -1) & (y_true == 0)))

    print(f"{title} 3-way policy:")
    print("  hard FN (recyclable predicted non-recyclable):", hard_fn)
    print("  hard FP (non-recyclable predicted recyclable):", hard_fp)
    print("  uncertain recyclables:", uncertain_rec)
    print("  uncertain non-recyclables:", uncertain_non)
    print("  coverage (not uncertain):", float(np.mean(pred3 != -1)))

    # Accepted-only confusion matrix/report (only where pred != -1)
    mask = pred3 != -1
    if np.sum(mask) == 0:
        print("No accepted predictions under current thresholds.")
        return

    y_true_acc = y_true[mask]
    y_pred_acc = pred3[mask]

    print("\nAccepted-only classification report:")
    print(classification_report(
        y_true_acc, y_pred_acc,
        target_names=["non-recyclable", "recyclable"],
        digits=4
    ))

    cm = confusion_matrix(y_true_acc, y_pred_acc)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix {title} (accepted only)")
    plt.colorbar()
    plt.xticks([0, 1], ["non-recyclable", "recyclable"], rotation=30)
    plt.yticks([0, 1], ["non-recyclable", "recyclable"])
    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

# Requires real_life_ds and your calibrated thresholds.
# If your calibration stored them in best dict, use best["t_pos"], best["t_neg"].
eval_stage1_three_way(real_life_ds, t_pos=best["t_pos"], t_neg=best["t_neg"], title="REAL-LIFE")

In [ ]:
def eval_binary_hard(ds, threshold=0.5, title=""):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        p = model.predict(xb, verbose=0).reshape(-1)
        y_prob.append(p)
        y_true.append(tf.reshape(yb, [-1]).numpy())

    y_true = np.concatenate(y_true).astype(int)
    y_prob = np.concatenate(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    print(title)
    print(classification_report(
        y_true, y_pred,
        target_names=["non-recyclable", "recyclable"],
        digits=4
    ))

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix {title} (thr={threshold:.2f})")
    plt.colorbar()
    plt.xticks([0, 1], ["non-recyclable", "recyclable"], rotation=30)
    plt.yticks([0, 1], ["non-recyclable", "recyclable"])
    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

# Test set (pick 0.5 or your calibrated single-threshold if you have one)
eval_binary_hard(test_ds, threshold=0.5, title="TEST")

In [ ]:
print("\nDataset test evaluation:")
model.evaluate(test_ds, verbose=2)

print("\nReal-life evaluation:")
model.evaluate(real_life_ds, verbose=2)

In [ ]:
def collect_probs_and_labels(ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        p = model.predict(xb, verbose=0).reshape(-1)
        y_prob.append(p)
        y_true.append(tf.cast(tf.reshape(yb, [-1]), tf.float32).numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    return y_true, y_prob

y_true_rl, y_prob_rl = collect_probs_and_labels(real_life_ds)

print("Real-life samples:", len(y_true_rl))
print("Prob range:", float(np.min(y_prob_rl)), "to", float(np.max(y_prob_rl)))

In [ ]:
def calibrate_two_thresholds(
    y_true,
    y_prob,
    t_pos_grid=np.linspace(0.30, 0.90, 25),
    t_neg_grid=np.linspace(0.10, 0.70, 25),
    min_recyclable_recall=0.95,   # strict; relax to 0.90 if coverage too low
    min_nonrec_precision=0.80,    # how safe "non-recyclable" predictions must be
):
    """
    Policy:
      if p >= t_pos => recyclable (1)
      elif p <= t_neg => non-recyclable (0)
      else => uncertain (-1)

    Because your worst error is recyclable -> non-recyclable,
    we enforce high recall for recyclable (counting only confident recyclable as recall success).
    """
    best = None

    for t_pos in t_pos_grid:
        for t_neg in t_neg_grid:
            if t_neg >= t_pos:
                continue

            pred = np.full_like(y_prob, fill_value=-1, dtype=np.int32)
            pred[y_prob >= t_pos] = 1
            pred[y_prob <= t_neg] = 0

            coverage = float(np.mean(pred != -1))
            uncertain_rate = float(np.mean(pred == -1))

            # recyclable recall: among ALL true recyclables, how many predicted recyclable
            true_rec = (y_true == 1)
            if np.any(true_rec):
                recyclable_recall = float(np.mean(pred[true_rec] == 1))
            else:
                recyclable_recall = 0.0

            # non-recyclable precision: among predicted non-recyclable, how many truly non-recyclable
            pred_non = (pred == 0)
            nonrec_precision = float(np.mean(y_true[pred_non] == 0)) if np.any(pred_non) else 0.0

            # recyclable precision (nice to monitor)
            pred_rec = (pred == 1)
            rec_precision = float(np.mean(y_true[pred_rec] == 1)) if np.any(pred_rec) else 0.0

            if recyclable_recall < min_recyclable_recall:
                continue
            if nonrec_precision < min_nonrec_precision:
                continue

            candidate = {
                "t_pos": float(t_pos),
                "t_neg": float(t_neg),
                "coverage": coverage,
                "uncertain_rate": uncertain_rate,
                "recyclable_recall": recyclable_recall,
                "nonrec_precision": nonrec_precision,
                "recyclable_precision": rec_precision,
                "pred_rec_rate": float(np.mean(pred == 1)),
                "pred_nonrec_rate": float(np.mean(pred == 0)),
            }

            # maximize coverage, then recyclable_precision
            if best is None:
                best = candidate
            else:
                if candidate["coverage"] > best["coverage"] + 1e-12:
                    best = candidate
                elif abs(candidate["coverage"] - best["coverage"]) < 1e-12 and candidate["recyclable_precision"] > best["recyclable_precision"]:
                    best = candidate

    return best

best = calibrate_two_thresholds(
    y_true_rl,
    y_prob_rl,
    min_recyclable_recall=0.95,
    min_nonrec_precision=0.80,
)

if best is None:
    print("No thresholds satisfied constraints. Try relaxing constraints, e.g.:")
    print("  min_recyclable_recall=0.90 and/or min_nonrec_precision=0.70")
else:
    print("Chosen thresholds (real-life calibrated):")
    print(f"  t_pos (recyclable if p>=t_pos): {best['t_pos']:.2f}")
    print(f"  t_neg (non-recyclable if p<=t_neg): {best['t_neg']:.2f}")
    print(f"  coverage (not uncertain): {best['coverage']:.3f}")
    print(f"  uncertain_rate: {best['uncertain_rate']:.3f}")
    print(f"  recyclable_recall: {best['recyclable_recall']:.3f}")
    print(f"  nonrec_precision: {best['nonrec_precision']:.3f}")
    print(f"  recyclable_precision: {best['recyclable_precision']:.3f}")
    print(f"  pred_rec_rate: {best['pred_rec_rate']:.3f}")
    print(f"  pred_nonrec_rate: {best['pred_nonrec_rate']:.3f}")

In [ ]:
def apply_three_way_policy(y_prob, t_pos, t_neg):
    pred = np.full_like(y_prob, fill_value=-1, dtype=np.int32)
    pred[y_prob >= t_pos] = 1
    pred[y_prob <= t_neg] = 0
    return pred

if best is not None:
    pred3 = apply_three_way_policy(y_prob_rl, best["t_pos"], best["t_neg"])

    # Report hard errors (especially recyclable predicted non-recyclable)
    hard_fn = np.sum((pred3 == 0) & (y_true_rl == 1))   # recyclable -> non-recyclable (worst)
    hard_fp = np.sum((pred3 == 1) & (y_true_rl == 0))   # non-recyclable -> recyclable
    uncertain_rec = np.sum((pred3 == -1) & (y_true_rl == 1))
    uncertain_non = np.sum((pred3 == -1) & (y_true_rl == 0))

    print("3-way policy on real-life set:")
    print("  hard FN (recyclable predicted non-recyclable):", int(hard_fn))
    print("  hard FP (non-recyclable predicted recyclable):", int(hard_fp))
    print("  uncertain recyclables:", int(uncertain_rec))
    print("  uncertain non-recyclables:", int(uncertain_non))

In [ ]:
OUT_DIR = "./stage1_mobilenetv3large"
os.makedirs(OUT_DIR, exist_ok=True)

model_path = os.path.join(OUT_DIR, "model.keras")
model.save(model_path)
print("Saved model to:", model_path)

if best is not None:
    thr_path = os.path.join(OUT_DIR, "thresholds.npz")
    np.savez(thr_path, t_pos=best["t_pos"], t_neg=best["t_neg"])
    print("Saved thresholds to:", thr_path)